In [1]:
# This function takes a date input with the string format 'YYYYMMDD' 
# Searches for the DIMM and MASS URL that is associated with the date
# Compares the two files and matches the MASS entries with the closest DIMM entries by time
# And creates a dataframe with 6 columns of data:

# midpoint_time (HST) : The time between DIMM and MASS measurements
# dimm_time (HST) : The time the DIMM measurement was taken 
# mass_time (HST) : The time the MASS measurement was taken 
# dimm_value : The measurement taken at the specific DIMM time (arcseconds)
# mass_value : The measurement taken at the specific MASS time (arcseconds)
# time_difference (sec) : delta(T) between the DIMM and MASS time 

# Then extracts the year (YYYY) from the original input
# Searches for the CFHT URL that is accociated with the year
# Compares the CFHT data with the 6-column dataframe 
# Matches the midpoint_time (HST) from the dataframe to the closest CHFT measurement
# And creates a new dataframe with the original 6 columns of data plus 7 more from the CFHT dataset
# The new 7 columns are:

# CFHT_datetime (HST) : The closest time the CFHT measurement was taken
# CHFT_windspeed (m/s) : The windspeed at that time
# CHFT_wind_direction (dec) : The wind direction from East to West at that time 
# CHFT_temp (°C) : Temperature at that time
# CFHT_humidity (%)': Humidity at that time
# CFHT_pressure (Pa)': Pressure at that time
# CFHT_time_diff (from midpoint) (sec)' delta(T) between the midpoint_time (HST) and CFHT_datetime (HST)

# Removes any rows with NaN values
# And returns a 13-column dataframe with the filename: '{date}_result.csv'

# *Note* : Must use these imports to run function: 
# import pandas as pd
# import numpy as np
# from datetime import datetime, timedelta
# import requests

def make_dat_file(date: str):

    # Check if date is proper format 
    try:
        datetime.strptime(date, '%Y%m%d')
    except ValueError:
        print("Invalid date format. Use YYYYMMDD.")
        return

    # Define the URLs for DIMM and MASS
    dimm_url = f"http://mkwc.ifa.hawaii.edu/current/seeing/dimm//{date}.dimm.dat"
    mass_url = f"http://mkwc.ifa.hawaii.edu/current/seeing/mass//{date}.mass.dat"
    
    # Extract the year from the date string
    year = date[:4]

    # Define the URL for CFHT
    cfht_url = f"http://mkwc.ifa.hawaii.edu/archive/wx/cfht/cfht-wx.{year}.dat"

    # Check if DIMM and MASS files are valid size (above 0 bytes)
    # DIMM check
    dimm_response = requests.get(dimm_url)
    dimm_file_size = len(dimm_response.content)
        
    if dimm_file_size < 1:
        return
            
    #MASS check
    mass_response = requests.get(mass_url)
    mass_file_size = len(mass_response.content)
        
    if mass_file_size < 1:
        return

    # Load the data files
    dimm_data = pd.read_csv(dimm_url, sep=r'\s+', header=None)
    mass_data = pd.read_csv(mass_url, sep=r'\s+', header=None)

    # Define DIMM and MASS column names
    columns = ['year', 'month', 'day', 'hour', 'minute', 'second', 'value']

    # Define CFT column names
    cfht_cols = ['year', 'month', 'day', 'hour', 'minute', 
                 'wind_speed', 'wind_dir', 'temp', 'humidity', 'pressure']

    # Assign column names to the dataframes
    mass_data.columns = columns
    dimm_data.columns = columns

    # Convert to datetime
    mass_data['datetime'] = pd.to_datetime(mass_data[['year', 'month', 'day', 'hour', 'minute', 'second']])
    dimm_data['datetime'] = pd.to_datetime(dimm_data[['year', 'month', 'day', 'hour', 'minute', 'second']])

    # Sort data
    mass_data.sort_values('datetime', inplace=True)
    dimm_data.sort_values('datetime', inplace=True)

    # Match closest times
    closest_matches = []
    for _, dimm_row in dimm_data.iterrows():
        dimm_time = dimm_row['datetime']
        time_diffs = np.abs((mass_data['datetime'] - dimm_time).dt.total_seconds())
        closest_idx = time_diffs.idxmin()
        mass_row = mass_data.loc[closest_idx]

        midpoint_time = dimm_time + (mass_row['datetime'] - dimm_time) / 2
        time_difference = abs((mass_row['datetime'] - dimm_time).total_seconds())
    
        closest_matches.append({
            'dimm_time (HST)': dimm_time,
            'mass_time (HST)': mass_row['datetime'],
            'midpoint_time (HST)': midpoint_time,
            'dimm_value': dimm_row['value'],
            'mass_value': mass_row['value'],
            'time_difference (sec)': time_difference
        })

    # Create final comparison DataFrame
    comparison_df = pd.DataFrame(closest_matches)

    # Reorder the columns to have dimm_time, mass_time, then midpoint_time
    cols = list(comparison_df.columns)
    
    # Move dimm_time and mass_time before midpoint_time
    for col in ['mass_time (HST)', 'dimm_time (HST)']:
        cols.insert(cols.index('midpoint_time (HST)'), cols.pop(cols.index(col)))

    comparison_df = comparison_df[cols]

    # Round up midpoint_time to the next full second
    comparison_df['midpoint_time (HST)'] = comparison_df['midpoint_time (HST)'].apply(
        lambda dt: dt if dt.microsecond == 0 else pd.to_datetime(np.ceil(dt.value / 1e9) * 1e9)
    )

    # Load CFHT weather data and assign column names 
    cfht_data = pd.read_csv(
        cfht_url,
        sep=r'\s+',
        names=cfht_cols,
        comment='#',
        header=None,
        on_bad_lines='skip'
    )

    # Remove any rows with NaNs in datetime components
    cfht_data.dropna(subset=['year', 'month', 'day', 'hour', 'minute'], inplace=True)

    # Ensure datetime columns are integers
    cfht_data[['year', 'month', 'day', 'hour', 'minute']] = cfht_data[['year', 'month', 'day', 'hour', 'minute']].astype(int)

    # Build datetime safely
    cfht_data['datetime'] = pd.to_datetime(cfht_data[['year', 'month', 'day', 'hour', 'minute']], errors='coerce')

    # Drop rows where datetime conversion failed
    cfht_data.dropna(subset=['datetime'], inplace=True)

    # Drop previous CFT columns if they exist
    cfht_cols = [col for col in comparison_df.columns if col.startswith('CFT_')]
    comparison_df = comparison_df.drop(columns=cfht_cols, errors='ignore')

    # Match CFHT weather to midpoint times
    def find_closest_cfht(row):
        diffs = np.abs((cfht_data['datetime'] - row['midpoint_time (HST)']).dt.total_seconds())
        closest_idx = diffs.idxmin()
        closest_row = cfht_data.loc[closest_idx]
        return pd.Series({
            'CFHT_datetime (HST)': closest_row['datetime'],
            'CFHT_windspeed (m/s)': closest_row['wind_speed'],
            'CFHT_wind_direction (dec)': closest_row['wind_dir'],
            'CFHT_temp (°C)': closest_row['temp'],
            'CFHT_humidity (%)': closest_row['humidity'],
            'CFHT_pressure (Pa)': closest_row['pressure'],
            'CFHT_time_diff (from midpoint) (sec)': diffs[closest_idx]
        })

    # Apply fresh match function and combine
    cfht_matches = comparison_df.apply(find_closest_cfht, axis=1)
    comparison_df = pd.concat([comparison_df, cfht_matches], axis=1)

    # Drop rows with any NaN values and reassign
    comparison_df = comparison_df.dropna()

    # Convert windspeed column from knots to m/s 
    comparison_df.iloc[:, 7] = comparison_df.iloc[:, 7] * 0.514444

    # Convert pressure column from mbar to Pa
    comparison_df.iloc[:, 11] = comparison_df.iloc[:, 11] * 100

    # Write results to separate data file
    comparison_df.to_csv(f"{date}_result.csv", index=False)

    return